In [134]:
import pandas as pd
import numpy as np

### Problem a

##### Read data from learningOutcomes.csv

In [135]:
outcomes = pd.read_csv('learningOutcomes.csv', names = ['ID', 'Activity', "Score"], header=None)
outcomes.head()

,ID,Activity,Score
0,5410951,activity2,53
1,5410952,activity2,79
2,5410953,activity1,207
3,5410954,activity1,179
4,5410955,activity2,68


##### Calculate means

In [136]:
mean_1 = outcomes['Score'].where(outcomes['Activity'] == 'activity1').mean()
mean_2 = outcomes['Score'].where(outcomes['Activity'] == 'activity2').mean()
mean_diff = abs(mean_1 - mean_2)
print(f"""
E1 = {mean_1}
E2 = {mean_2}
E1 - E2 = {mean_diff}
""")


E1 = 144.9280442804428
E2 = 153.12941176470588
E1 - E2 = 8.201367484263073



##### Calculate the number of activity 1 and activity 2 in the sample data

In [137]:
n_1 = outcomes['Score'].where(outcomes['Activity'] == 'activity1').count()
n_2 = outcomes['Score'].where(outcomes['Activity'] == 'activity2').count()
print(f"""
N1 = {n_1}
N2 = {n_2}
""")


N1 = 542
N2 = 510



##### Explanation

Select the rows with Activity equals to activity1 and activity2, and calculte the mean of the score seperatly

### Question b

##### Null hypothesis: 
There's no difference between the distribution of score for activity 1 and activity 2

##### Calculate p-value

In [138]:
# Calculate the PMF of score under null hypothesis
pmf = outcomes['Score'].value_counts().sort_index() / outcomes['Score'].value_counts().sum()
pmf

Score
8      0.002852
10     0.000951
11     0.003802
16     0.000951
17     0.002852
         ...   
329    0.000951
342    0.000951
347    0.000951
371    0.000951
390    0.000951
Name: count, Length: 267, dtype: float64

In [139]:
# Calculate p-value through running simulations
# Initiate count for the number of simulations with exceeded mean difference
count = 0
# Run simulations 1000 times
for _ in range (1000):
    # Draw 542 and 510 samples of activity 1 and activity 2 from the distribution
    sample_1 = pd.Series(np.random.choice(a=pmf.index, p=pmf.values, size=n_1))
    sample_2 = pd.Series(np.random.choice(a=pmf.index, p=pmf.values, size=n_2))
    # Test if the difference of observed mean difference exceeds the calculated mean difference
    if abs(sample_1.mean() - sample_2.mean()) > mean_diff:
        count += 1
p_value = count / 1000
print(f"p-value = {p_value}")

p-value = 0.06


### Question 3

##### Read data from background.csv

In [140]:
background = pd.read_csv('background.csv', names=['ID', 'Experience'], header=None)
background.head()

,ID,Experience
0,5410951,less
1,5410952,more
2,5410953,more
3,5410954,average
4,5410955,less


##### Build helper function to calculate mean difference and p-value

In [141]:
def helper(experience):
    # Find the ids correspond to the given experience
    ids = background.loc[background['Experience'] == experience, 'ID']

    # Calculate the size
    n_1 = outcomes['Score'].loc[(outcomes['Activity'] == 'activity1') & (outcomes['ID'].isin(ids))].count()
    n_2 = outcomes['Score'].loc[(outcomes['Activity'] == 'activity2') & (outcomes['ID'].isin(ids))].count()

    # Calculate the mean difference
    mean_1 = outcomes['Score'].loc[(outcomes['Activity'] == 'activity1') & (outcomes['ID'].isin(ids))].mean()
    mean_2 = outcomes['Score'].loc[(outcomes['Activity'] == 'activity2') & (outcomes['ID'].isin(ids))].mean()
    mean_diff = abs(mean_1 - mean_2)

    # Calculate p-value
    count = 0
    for _ in range (1000):
        sample_1 = pd.Series(np.random.choice(a=pmf.index, p=pmf.values, size=n_1))
        sample_2 = pd.Series(np.random.choice(a=pmf.index, p=pmf.values, size=n_2))
        if abs(sample_1.mean() - sample_2.mean()) > mean_diff:
            count += 1
    p_value = count / 1000

    return (mean_diff, p_value)

##### Use helper function to calculate the mean difference and p-value for each background

In [142]:
for experience in ('less', 'average', 'more'):
    mean_diff, p_value = helper(experience)
    print(f"For {experience} experience students, mean_diff = {mean_diff}, p_value = {p_value}")

For less experience students, mean_diff = 26.015873015873012, p_value = 0.001
For average experience students, mean_diff = 24.984501450603148, p_value = 0.004
For more experience students, mean_diff = 28.416450216450215, p_value = 0.0
